In [ ]:
import Modules.profiledatamanager as NFS_PM
import pandas as pd
import Modules.reportinformation as NFS_RI

df_caseinfo = pd.read_csv('./testdata/test_df_caseinfo.csv')
df_report = pd.read_csv('./testdata/test_df_report.csv')
df_str = pd.read_csv('./testdata/test_df_profile_str.csv')
df_ystr = pd.read_csv('./testdata/test_df_profile_ystr.csv')
pm_str = NFS_PM.NFSProfileDataManager(kit="STR")
pm_str.df_profile = df_str
pm_ystr = NFS_PM.NFSProfileDataManager(kit='YSTR')
pm_ystr.df_profile = df_ystr

In [ ]:
info = NFS_RI.NFSReportInformation(id_case='2025-C-6697')
info.extract_caseinfo_from_df(df_caseinfo)
info.extract_evidenceinfo_from_df(df_report)
info.load_str_profiledatamanager(pm_str)
info.load_ystr_profiledatamanager(pm_ystr)

In [ ]:
from typing import Dict, List
import pandas as pd

class NFS_ReportWriter():
    """NFSReportInformation 인스턴스의 데이터를 토대로 HWP control을 사용하여 감정서를 작성하는 클래스"""
    def __init__(self, ReportData:NFS_RI.NFSReportInformation, paths_picture:list):
        self.ReportData = ReportData
        self.paths_picture=paths_picture

        self.code_categorized_str = None
        self.df_str_indexed = None
        self.code_categorized_ystr = None
        self.df_ystr_indexed = None
    
    def categorize_df(self) -> None:
        """ReportData.df_evidences를 유형별로 분류하여 정리.
        
        - code_categorized: 코드-프로필_유형을 키-밸류로 정리하여 딕셔너리화
        - df_str_indexed: ReportData.df_evidences를 MultiIndex로 설정하여 빠른 조회 가능
        """
    
        def categorize_code(df: pd.DataFrame, column_prefix: str = "") -> Dict[str, List[str]]:
            """코드를 프로필 유형에 따라 분류.
            
            Args:
                df: 유형별 분류할 리포트 데이터프레임
                column_prefix: 컬럼명 접두사 (Y-STR의 경우 'Y_')
                
            Returns:
                프로필 유형별로 그룹화된 코드 딕셔너리 e.g {V: ['2025-C-3123-1'], S: ['2025-3123-2']}
            """

            code_col = f"{column_prefix}코드"
            profile_col = f"{column_prefix}프로필_유형"

            return df.groupby(profile_col)[code_col].unique().to_dict()
    
        # STR 데이터 처리
        df_str = self.ReportData.evidenceinfo[self.ReportData.evidenceinfo["기재_여부"] == "기재"].copy()
        self.code_categorized_str = categorize_code(df_str)
        
        # MultiIndex 설정 (기존 groupby 대체)
        self.df_str_indexed = df_str.set_index(["코드", "프로필_유형"])
        self.df_str_indexed.sort_index(inplace=True)  # 조회 성능 향상을 위한 정렬
        
        # Y-STR 데이터 처리
        df_ystr = self.ReportData.evidenceinfo[self.ReportData.evidenceinfo["Y_기재_여부"] == "기재"].copy()
        self.code_categorized_ystr = categorize_code(df_ystr, column_prefix="Y_")
        
        # MultiIndex 설정 (기존 groupby 대체)
        self.df_ystr_indexed = df_ystr.set_index(["Y_코드", "Y_프로필_유형"])
        self.df_ystr_indexed.sort_index(inplace=True)  # 조회 성능 향상을 위한 정렬
        
    